# VIS4AI · Notebook 01 — How does a small reasoning model solve GSM8K, and when does it "know" the answer?

**Model:** `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B` (switchable via one config variable)
**Data:** GSM8K test split + Apple's GSM-Symbolic perturbations (+ our own "irrelevant sentence" perturbation)

This notebook answers two questions:

1. **How does the model behave?** Accuracy, reasoning-chain length, backtracking ("wait", "alternatively", …) and robustness to perturbed problems.
2. **What did the model learn?** With *early answering* (cut the reasoning short and force an answer) and a *logit lens* (read intermediate layers), we look at **when** during the chain of thought the model already "knows" the final answer.

### How to run
1. `Runtime → Change runtime type → T4 GPU`.
2. `Runtime → Run all`. You will be asked to authorise Google Drive once.
3. **If the session disconnects:** reconnect and simply *Run all* again. Every result is appended to JSONL files on Drive after each batch, and finished problems are skipped, so the notebook resumes where it stopped.

### Sections
1. Setup (packages, GPU, Drive, config)
2. Load model and tokenizer
3. Load data (GSM8K, GSM-Symbolic, own perturbation)
4. Prompting
5. Answer extraction (+ unit tests) — *defined before the generation loop, because the loop uses it*
6. Generation loop (batched, resumable)
7. Behaviour metrics and plots
8. "What did it learn?" — early answering
9. Optional: logit lens
10. Next steps

## 1. Setup

We install/upgrade the Hugging Face stack (Colab already ships PyTorch), check the GPU, mount Google Drive and define **all configuration in one cell**. To switch to the baseline model later, change only `MODEL_KEY`.

In [ ]:
# Colab already has torch; make sure the HF libraries are recent enough.
# (Pip may print dependency warnings about other preinstalled Colab packages — they are harmless here.)
!pip install -q "transformers>=4.45" "accelerate>=0.33" "datasets>=2.20"

In [ ]:
import os, re, json, math, random, time, gc, hashlib, subprocess
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

print(f"torch {torch.__version__} | transformers {transformers.__version__}")

if torch.cuda.is_available():
    DEVICE = "cuda"
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | {props.total_memory / 1e9:.1f} GB | compute capability {props.major}.{props.minor}")
    # The T4 (compute capability 7.5) has no bfloat16 support -> we use float16 everywhere.
    print("bfloat16 supported:", torch.cuda.is_bf16_supported())
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv"],
                         capture_output=True, text=True).stdout)
else:
    DEVICE = "cpu"
    print("WARNING: no GPU found. Use Runtime -> Change runtime type -> T4 GPU. (CPU only works for tiny smoke tests.)")

In [ ]:
# Mount Google Drive so results survive disconnects. Outside Colab we fall back to a local folder.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("Running in Colab:", IN_COLAB)

### Configuration
Everything you might want to change lives in the next cell.

* `MODEL_KEY` picks an entry from `MODEL_PRESETS` (the baseline `Qwen/Qwen2.5-Math-1.5B` is already prepared for Notebook 02).
* Results go to `DRIVE_ROOT/results/<RUN_NAME>/`. The run name contains model, decoding mode, token budget and seed, so different settings never overwrite each other — and re-running with the same settings **resumes**.
* Start small (`N_PROBLEMS = 50`). Increasing it later only generates the new problems.

In [ ]:
# ---------------------------------------------------------------- model choice
MODEL_PRESETS = {
    # Main model: an R1-distilled reasoning model. Writes <think> ... </think> before the answer.
    "r1-distill-qwen-1.5b": dict(name="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B", is_reasoning=True),
    # Baseline for Notebook 02: same size / base family, but no R1-style long reasoning.
    "qwen2.5-math-1.5b": dict(name="Qwen/Qwen2.5-Math-1.5B", is_reasoning=False),
}
MODEL_KEY = "r1-distill-qwen-1.5b"          # <- the single switch for the model

# ---------------------------------------------------------------- generation
N_PROBLEMS = 50            # GSM8K problems per split (start small, increase later)
MAX_NEW_TOKENS = 2048      # token budget for reasoning + answer
BATCH_SIZE = 16            # T4: 16 is safe for a 1.5B model at 2048 new tokens (auto-halves on OOM)
GREEDY = False             # False = DeepSeek's recommended sampling, True = greedy decoding
TEMPERATURE = 0.6          # DeepSeek recommendation for R1 distills
TOP_P = 0.95               # DeepSeek recommendation for R1 distills
SEED = 42

# Which problem sets to generate (in this order). Remove entries to save time.
#   gsm8k        original GSM8K test problems
#   symbolic     GSM-Symbolic "main": same template, new names + numbers (new gold answer)
#   symbolic_p1  GSM-Symbolic "P1": one extra clause (harder)
#   noop         our own perturbation: one irrelevant sentence inserted (gold answer unchanged)
SPLITS_TO_RUN = ["gsm8k", "symbolic", "symbolic_p1", "noop"]
SYMBOLIC_INSTANCE = 0      # GSM-Symbolic has 50 generated instances per template; we use one

USE_VLLM = False           # optional, experimental (see Section 6). HF transformers is the default.

# ---------------------------------------------------------------- "what did it learn" experiments
N_EARLY = 10                                   # problems for early answering
EARLY_FRACTIONS = [round(x, 2) for x in np.linspace(0.0, 1.0, 11)]   # 0%, 10%, ..., 100% of the reasoning
RUN_LOGIT_LENS = True                          # Section 9 (optional)
N_LOGIT_LENS = 4                               # problems for the logit-lens heatmaps

# ---------------------------------------------------------------- paths
DRIVE_ROOT = "/content/drive/MyDrive/VIS4AI" if IN_COLAB else "./VIS4AI"

# ------------------------------------------------- derived values (no need to edit below)
MODEL_NAME = MODEL_PRESETS[MODEL_KEY]["name"]
IS_REASONING_MODEL = MODEL_PRESETS[MODEL_KEY]["is_reasoning"]
DECODING = "greedy" if GREEDY else f"t{TEMPERATURE}-p{TOP_P}"
RUN_NAME = f"{MODEL_KEY}__{DECODING}__{MAX_NEW_TOKENS}tok__seed{SEED}"
OUT_DIR = Path(DRIVE_ROOT) / "results" / RUN_NAME
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = dict(model_key=MODEL_KEY, model_name=MODEL_NAME, n_problems=N_PROBLEMS, max_new_tokens=MAX_NEW_TOKENS,
              batch_size=BATCH_SIZE, greedy=GREEDY, temperature=TEMPERATURE, top_p=TOP_P, seed=SEED,
              splits=SPLITS_TO_RUN, symbolic_instance=SYMBOLIC_INSTANCE, n_early=N_EARLY,
              early_fractions=EARLY_FRACTIONS)
with open(OUT_DIR / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

set_seed(SEED)
print("Results folder:", OUT_DIR)

## 2. Load model and tokenizer

The model is loaded in **float16** (the T4 cannot do bfloat16). A 1.5B model needs about 3.5 GB of VRAM, which leaves plenty of room for the KV cache during batched generation.
We use **left padding**, which is required for batched generation with decoder-only models (all prompts must end at the same position).

In [ ]:
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32   # float32 only for CPU smoke tests

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=DTYPE, attn_implementation="sdpa")
model = model.to(device=DEVICE, dtype=DTYPE).eval()

# All token ids that end a generation (the generation config may list several).
_eos = model.generation_config.eos_token_id
EOS_IDS = set(_eos if isinstance(_eos, (list, tuple)) else [_eos]) | {tokenizer.eos_token_id}
# Qwen chat models end a turn with <|im_end|>; base models like Qwen2.5-Math may not list it as EOS.
_im_end = tokenizer.convert_tokens_to_ids("<|im_end|>")
if isinstance(_im_end, int) and _im_end != tokenizer.unk_token_id:
    EOS_IDS.add(_im_end)

print(f"Loaded {MODEL_NAME} | dtype {model.dtype} | {model.num_parameters() / 1e9:.2f} B parameters")
print(f"Model weights: {model.get_memory_footprint() / 1e9:.2f} GB")
if DEVICE == "cuda":
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB "
          f"(reserved {torch.cuda.memory_reserved() / 1e9:.2f} GB)")

## 3. Load the data

* **GSM8K** (`openai/gsm8k`, config `main`, split `test`, 1,319 problems). The gold answer is the number after `####`.
* **GSM-Symbolic** (`apple/GSM-Symbolic`, configs `main`, `p1`, `p2`, split `test`). Apple built 100 templates from GSM8K test problems and generated 50 instances per template; `original_id` points back to the GSM8K test index. *P1/P2 add one/two extra clauses (harder).* If the Hugging Face download fails, we fall back to the raw JSONL files in Apple's GitHub repo, and if that fails too, the symbolic splits are skipped.
* **NoOp (own)**: Apple's paper also used "GSM-NoOp" (an added irrelevant sentence), but that set was not released. We build a simple version ourselves: we insert one irrelevant sentence before the final question. The gold answer stays the same.

**Problem selection:** if GSM-Symbolic is available, we first pick GSM8K problems that *have* a GSM-Symbolic template, so every original problem has a perturbed twin and we can compare them pairwise.

In [ ]:
from datasets import load_dataset

def parse_gold(answer_field: str) -> str:
    # GSM8K answers end with '#### <number>'; strip thousands separators.
    return answer_field.split("####")[-1].strip().replace(",", "")

gsm8k = load_dataset("openai/gsm8k", "main", split="test")
print(f"GSM8K test: {len(gsm8k)} problems")
print("Example gold answers:", [parse_gold(gsm8k[i]["answer"]) for i in range(5)])

In [ ]:
SYMBOLIC_GITHUB = "https://raw.githubusercontent.com/apple/ml-gsm-symbolic/main/generated_data/{}.jsonl"
SYMBOLIC_FILES = {"main": "GSM_symbolic", "p1": "GSM_p1", "p2": "GSM_p2"}

def load_gsm_symbolic(config: str):
    # Try Hugging Face first, then Apple's GitHub, otherwise return None.
    try:
        return load_dataset("apple/GSM-Symbolic", name=config, split="test").to_pandas()
    except Exception as e:
        print(f"[GSM-Symbolic/{config}] Hugging Face load failed: {type(e).__name__}: {e}")
    try:
        df = pd.read_json(SYMBOLIC_GITHUB.format(SYMBOLIC_FILES[config]), lines=True)
        print(f"[GSM-Symbolic/{config}] loaded from GitHub fallback")
        return df
    except Exception as e:
        print(f"[GSM-Symbolic/{config}] GitHub fallback failed: {type(e).__name__}: {e}")
    return None

symbolic = {cfg: load_gsm_symbolic(cfg) for cfg in ["main", "p1"]}
for cfg, df in symbolic.items():
    if df is not None:
        print(f"GSM-Symbolic {cfg}: {len(df)} rows, {df['id'].nunique()} templates, "
              f"{df['instance'].nunique()} instances per template")

# Sanity check: original_id really indexes the GSM8K test split.
if symbolic["main"] is not None:
    row = symbolic["main"].iloc[0]
    assert gsm8k[int(row["original_id"])]["question"].strip() == row["original_question"].strip()
    print("original_id -> GSM8K test index: OK")

In [ ]:
# ---- Own perturbation: insert an irrelevant sentence before the final question.
# The sentences are deliberately unrelated to any quantity asked for, so the gold answer does not change.
NOOP_SENTENCES = [
    "The weather that day was sunny and pleasant.",
    "Their neighbor recently painted the fence blue.",
    "A local bakery opened 12 years ago on the same street.",
    "Most people in the town enjoy listening to music in the evening.",
    "The calendar on the wall shows a picture of 3 mountains.",
    "Someone mentioned that the library closes at 8 pm on Fridays.",
]

def add_irrelevant_sentence(question: str, rng: random.Random) -> str:
    # Split into sentences and insert a distractor right before the last one (usually the question).
    sentences = re.split(r"(?<=[.!?])\s+", question.strip())
    distractor = rng.choice(NOOP_SENTENCES)
    if len(sentences) == 1:
        return distractor + " " + sentences[0]
    return " ".join(sentences[:-1] + [distractor, sentences[-1]])

# ---- Select GSM8K problems (templated ones first, so originals and perturbations can be paired).
rng = random.Random(SEED)
templated_ids = sorted(set(symbolic["main"]["original_id"])) if symbolic["main"] is not None else []
other_ids = sorted(set(range(len(gsm8k))) - set(templated_ids))
rng.shuffle(templated_ids)
rng.shuffle(other_ids)
selected_ids = (templated_ids + other_ids)[:N_PROBLEMS]

# ---- Build one list of problem dicts per split. 'uid' is the unique key used for resuming.
PROBLEMS = {}
PROBLEMS["gsm8k"] = [dict(uid=f"gsm8k-{i}", split="gsm8k", gsm8k_id=i, question=gsm8k[i]["question"],
                          gold=parse_gold(gsm8k[i]["answer"])) for i in selected_ids]

noop_rng = random.Random(SEED + 1)
PROBLEMS["noop"] = [dict(uid=f"noop-{p['gsm8k_id']}", split="noop", gsm8k_id=p["gsm8k_id"],
                         question=add_irrelevant_sentence(p["question"], noop_rng), gold=p["gold"])
                    for p in PROBLEMS["gsm8k"]]

for split, cfg in [("symbolic", "main"), ("symbolic_p1", "p1")]:
    df = symbolic[cfg]
    if df is None:
        PROBLEMS[split] = []
        continue
    df = df[(df["instance"] == SYMBOLIC_INSTANCE) & (df["original_id"].isin(selected_ids))]
    PROBLEMS[split] = [dict(uid=f"{split}-{r.id}-{r.instance}", split=split, gsm8k_id=int(r.original_id),
                            question=r.question, gold=parse_gold(r.answer)) for r in df.itertuples()]

for split in SPLITS_TO_RUN:
    print(f"{split:12s}: {len(PROBLEMS.get(split, []))} problems")

print("\nExample NoOp perturbation:\n", PROBLEMS["noop"][0]["question"])

## 4. Prompting

DeepSeek's usage recommendations for the R1 distills:
* **no system prompt** — everything goes into the user message,
* append **"Please reason step by step, and put your final answer within \boxed{}."**,
* sample with **temperature 0.6, top-p 0.95** (greedy decoding can cause endless repetition; `GREEDY = True` is still available for deterministic runs),
* the response should start with `<think>\n`. The model's chat template already appends `<think>\n` to the generation prompt; we check this and add it ourselves if a template does not.

We use the tokenizer's chat template so the exact special tokens match training.

In [ ]:
INSTRUCTION = "Please reason step by step, and put your final answer within \\boxed{}."

def build_prompt(question: str) -> str:
    messages = [{"role": "user", "content": f"{question}\n{INSTRUCTION}"}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    if IS_REASONING_MODEL and not prompt.rstrip().endswith("<think>"):
        prompt += "<think>\n"
    return prompt

if GREEDY:
    # temperature/top_p=None silences warnings caused by the sampling defaults in generation_config.json
    GEN_KWARGS = dict(do_sample=False, temperature=None, top_p=None, top_k=None)
else:
    # top_k=0 disables HF's default top-k=50, so only top-p filtering is applied (as recommended).
    GEN_KWARGS = dict(do_sample=True, temperature=TEMPERATURE, top_p=TOP_P, top_k=0)
GEN_KWARGS.update(max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tokenizer.pad_token_id, eos_token_id=sorted(EOS_IDS))

example_prompt = build_prompt(PROBLEMS["gsm8k"][0]["question"])
print(example_prompt)
print("Prompt length (tokens):", len(tokenizer(example_prompt, add_special_tokens=False).input_ids))
print("Generation kwargs:", GEN_KWARGS)

## 5. Answer extraction

The model is asked to put its answer in `\boxed{...}`. We
1. split the output into **reasoning** (before `</think>`) and **final answer text** (after it),
2. take the **last** `\boxed{...}` (with balanced braces, so `\boxed{\frac{1}{2}}` works),
3. normalise the content to a number: remove `$`, `\%`, thousands separators, `\text{...}` units, LaTeX spacing; evaluate `\frac{a}{b}`; take the last number,
4. compare numerically with the gold answer.

If there is no `\boxed{}` we fall back to the last number in the final-answer text and record `extraction="fallback"`. Since the raw output is saved, extraction can be improved later and re-applied without re-generating (Section 7 does exactly that).

In [ ]:
def split_reasoning(output: str):
    # Return (reasoning, final_text, think_closed).
    if not IS_REASONING_MODEL:
        return output.strip(), output.strip(), True
    if "</think>" in output:
        reasoning, final = output.split("</think>", 1)
        return reasoning.replace("<think>", "").strip(), final.strip(), True
    # </think> never produced: usually the token budget ran out in the middle of thinking.
    return output.replace("<think>", "").strip(), "", False


def last_boxed(text: str):
    # Content of the last \boxed{...} / \fbox{...}, respecting nested braces.
    matches = list(re.finditer(r"\\(?:boxed|fbox)\s*", text))
    if not matches:
        return None
    i = matches[-1].end()
    if i < len(text) and text[i] == "{":
        depth = 0
        for j in range(i, len(text)):
            if text[j] == "{":
                depth += 1
            elif text[j] == "}":
                depth -= 1
                if depth == 0:
                    return text[i + 1:j]
        return None                                 # unbalanced (output cut off)
    m = re.match(r"[^\s$]+", text[i:])               # '\boxed 42' without braces
    return m.group(0) if m else None


_NUM = r"-?\d+(?:\.\d+)?"

def to_number(s):
    # Normalise an answer string like '$1,234.50', '20\%' or '\frac{3}{4}' to a float (or None).
    if s is None:
        return None
    s = s.replace("−", "-").replace("\\!", "").replace("\\,", "").replace("\\;", "").replace("~", " ")
    s = re.sub(r"\\(?:text|mathrm|textbf|mbox)\{([^{}]*)\}", r" \1 ", s)      # keep the text, drop the command
    s = re.sub(r"\\[dt]?frac\{([^{}]*)\}\{([^{}]*)\}", r"(\1)/(\2)", s)        # \frac{a}{b} -> (a)/(b)
    s = s.replace("\\$", " ").replace("$", " ").replace("\\%", " ").replace("%", " ")
    s = re.sub(r"(?<=\d),(?=\d{3}(?!\d))", "", s)                              # 1,234 -> 1234
    frac = re.fullmatch(rf"\s*\(?({_NUM})\)?\s*/\s*\(?({_NUM})\)?\s*", s)
    if frac:
        num, den = float(frac.group(1)), float(frac.group(2))
        return num / den if den != 0 else None
    nums = re.findall(_NUM, s)
    return float(nums[-1]) if nums else None


def extract_answer(output: str):
    # Returns (value or None, method). method in {'boxed', 'fallback', 'none'}.
    _, final, closed = split_reasoning(output)
    text = final if (closed and final) else output   # if </think> is missing, look at everything
    boxed = last_boxed(text)
    if boxed is not None and to_number(boxed) is not None:
        return to_number(boxed), "boxed"
    if closed and final:
        m = re.findall(rf"answer is[^\d-]*({_NUM})", final.replace(",", ""))
        nums = m or re.findall(_NUM, final.replace(",", ""))
        if nums:
            return float(nums[-1]), "fallback"
    return None, "none"


def is_correct(pred, gold) -> bool:
    g = to_number(str(gold))
    return pred is not None and g is not None and math.isclose(pred, g, rel_tol=1e-6, abs_tol=1e-6)


def format_number(x) -> str:
    # 18.0 -> '18', 2.5 -> '2.5'; the exact string the model should write inside \boxed{}.
    x = float(x)
    return str(int(round(x))) if abs(x - round(x)) < 1e-9 else f"{x:.6f}".rstrip("0").rstrip(".")

### Unit tests for answer extraction
These run in a fraction of a second; if any assertion fails, fix the parser before spending GPU time.

In [ ]:
def _test_extraction():
    cases = [
        ("<think>...</think> The answer is \\boxed{18}.", 18),
        ("<think>...</think> \\boxed{1,234}", 1234),
        ("<think>...</think> \\boxed{\\$1,234.50}", 1234.5),
        ("<think>...</think> \\boxed{20\\%}", 20),
        ("<think>...</think> \\boxed{\\frac{3}{4}}", 0.75),
        ("<think>...</think> \\boxed{\\dfrac{10}{4}}", 2.5),
        ("<think>...</think> \\boxed{5 \\text{ dollars}}", 5),
        ("<think>...</think> \\boxed{x = -7}", -7),
        ("<think>first \\boxed{3}</think> then \\boxed{4}", 4),        # last box in the answer part wins
        ("<think>...</think> So she has 42 apples.", 42),              # fallback, no box
        ("<think>...</think> \\boxed 9", 9),
        ("<think>never finished, \\boxed{12", None),                   # cut off -> unbalanced
        ("<think>never finished, no answer", None),
    ]
    for text, expected in cases:
        pred, method = extract_answer(text)
        assert (pred is None and expected is None) or (pred is not None and math.isclose(pred, expected)), \
            f"{text!r}: got {pred} ({method}), expected {expected}"
    assert is_correct(18.0, "18") and is_correct(1234.0, "1,234") and not is_correct(None, "3")
    assert format_number(18.0) == "18" and format_number(2.5) == "2.5"
    assert split_reasoning("<think>\nabc\n</think>\n\nxyz") == ("abc", "xyz", True) or not IS_REASONING_MODEL
    print(f"All {len(cases)} extraction tests passed.")

_test_extraction()

## 6. Generation loop (batched and resumable)

* Results are written to `OUT_DIR/generations_<split>.jsonl`, **one line per problem, appended after every batch** (with `fsync`, so they reach Drive).
* On start we read the file and **skip problems whose `uid` is already saved**. A half-written last line from a disconnect is detected and removed.
* Prompts are sorted by length to reduce padding. If a batch runs out of GPU memory, it is split in half and retried.
* With HF `generate`, a batch is as slow as its longest answer — on a T4 expect roughly 1–2 minutes per batch of 16 at 2048 tokens.

Saved per problem: `uid, split, gsm8k_id, question, gold, output, reasoning, final_text, think_closed, pred, extraction, correct, n_tokens, n_reasoning_tokens, hit_limit` (+ decoding settings).

**Optional vLLM:** vLLM is much faster, but installing it in Colab replaces PyTorch and needs a runtime restart, and T4 support depends on the vLLM version. If you want to try: run `!pip install vllm` in a new cell, restart the runtime, set `USE_VLLM = True` and run all. If vLLM fails to import or start, the notebook silently falls back to HF transformers.

In [ ]:
def repair_jsonl(path: Path):
    # Remove a partially written last line (left behind if the runtime died mid-write).
    if not path.exists():
        return
    data = path.read_bytes()
    if data and not data.endswith(b"\n"):
        path.write_bytes(data[: data.rfind(b"\n") + 1])
        print(f"Repaired truncated last line in {path.name}")


def load_jsonl(path: Path):
    repair_jsonl(path)
    records = []
    if path.exists():
        with open(path) as f:
            for line in f:
                line = line.strip()
                if line:
                    try:
                        records.append(json.loads(line))
                    except json.JSONDecodeError:
                        print(f"Skipping corrupt line in {path.name}")
    return records


def append_jsonl(path: Path, records):
    with open(path, "a") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
        f.flush()
        os.fsync(f.fileno())

In [ ]:
def _postprocess(gen_ids):
    # Cut a generated id list at the first EOS; return (text, n_tokens, hit_limit).
    for k, t in enumerate(gen_ids):
        if t in EOS_IDS:
            return tokenizer.decode(gen_ids[:k], skip_special_tokens=True), k + 1, False
    return tokenizer.decode(gen_ids, skip_special_tokens=True), len(gen_ids), True


@torch.no_grad()
def generate_hf(prompts):
    enc = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
    try:
        out = model.generate(**enc, **GEN_KWARGS)
    except torch.cuda.OutOfMemoryError:
        del enc
        gc.collect(); torch.cuda.empty_cache()
        if len(prompts) == 1:
            raise
        print(f"  OOM with batch {len(prompts)} -> splitting in half")
        half = len(prompts) // 2
        return generate_hf(prompts[:half]) + generate_hf(prompts[half:])
    new_tokens = out[:, enc["input_ids"].shape[1]:].tolist()
    return [_postprocess(ids) for ids in new_tokens]


# ---- Optional vLLM backend (only used when USE_VLLM = True and vllm imports and starts).
vllm_engine = None
if USE_VLLM:
    try:
        from vllm import LLM, SamplingParams
        # Leave room on the GPU for the HF model, which Sections 8-9 still need.
        vllm_engine = LLM(model=MODEL_NAME, dtype="half", gpu_memory_utilization=0.55,
                          max_model_len=MAX_NEW_TOKENS + 1024, seed=SEED, enforce_eager=True)
        VLLM_PARAMS = SamplingParams(temperature=0.0 if GREEDY else TEMPERATURE,
                                     top_p=1.0 if GREEDY else TOP_P, max_tokens=MAX_NEW_TOKENS, seed=SEED,
                                     stop_token_ids=sorted(EOS_IDS))
        print("Using vLLM for generation.")
    except Exception as e:
        print(f"vLLM unavailable ({type(e).__name__}: {e}) -> falling back to HF transformers.")
        vllm_engine = None


def generate_vllm(prompts):
    outs = vllm_engine.generate(prompts, VLLM_PARAMS, use_tqdm=False)
    res = []
    for o in outs:
        c = o.outputs[0]
        res.append((c.text, len(c.token_ids), c.finish_reason == "length"))
    return res


def generate(prompts):
    return generate_vllm(prompts) if vllm_engine is not None else generate_hf(prompts)

In [ ]:
def make_record(problem, text, n_tokens, hit_limit):
    reasoning, final, closed = split_reasoning(text)
    pred, method = extract_answer(text)
    return dict(
        **problem,
        output=text, reasoning=reasoning, final_text=final, think_closed=closed,
        pred=pred, extraction=method, correct=is_correct(pred, problem["gold"]),
        n_tokens=n_tokens, n_reasoning_tokens=len(tokenizer(reasoning, add_special_tokens=False).input_ids),
        hit_limit=hit_limit, model=MODEL_NAME, greedy=GREEDY, temperature=TEMPERATURE, top_p=TOP_P,
        max_new_tokens=MAX_NEW_TOKENS, timestamp=time.strftime("%Y-%m-%d %H:%M:%S"),
    )


def run_split(split):
    path = OUT_DIR / f"generations_{split}.jsonl"
    done = {r["uid"] for r in load_jsonl(path)}
    pending = [p for p in PROBLEMS.get(split, []) if p["uid"] not in done]
    print(f"\n=== {split}: {len(done)} already saved, {len(pending)} to generate -> {path.name}")
    pending.sort(key=lambda p: len(p["question"]))      # similar lengths -> less padding
    step = len(pending) if vllm_engine is not None else BATCH_SIZE
    for b in range(0, len(pending), max(step, 1)):
        batch = pending[b:b + step]
        # Seed from the batch content, so a resumed run samples the same way for the same batch.
        set_seed(SEED + int(hashlib.md5(batch[0]["uid"].encode()).hexdigest()[:6], 16))
        t0 = time.time()
        outs = generate([build_prompt(p["question"]) for p in batch])
        records = [make_record(p, *o) for p, o in zip(batch, outs)]
        append_jsonl(path, records)
        dt = time.time() - t0
        n_tok = sum(r["n_tokens"] for r in records)
        print(f"  saved {len(done) + b + len(batch)}/{len(done) + len(pending)} | "
              f"batch acc {np.mean([r['correct'] for r in records]):.2f} | "
              f"hit limit {sum(r['hit_limit'] for r in records)} | {dt:.0f}s | {n_tok / dt:.0f} tok/s")


for split in SPLITS_TO_RUN:
    run_split(split)
print("\nGeneration finished.")

## 7. Behaviour metrics and plots

We reload everything from Drive (so this section also works on its own after a disconnect) and **re-apply answer extraction** to the raw outputs.

Metrics:
* **Accuracy** per problem set, with 95% Wilson confidence intervals (with 50 problems, CIs are wide — keep that in mind).
* **Reasoning length** (tokens before `</think>`), correct vs. incorrect.
* **Backtracking markers**: case-insensitive counts of "wait", "alternatively", "hmm", "let me check/verify", … in the reasoning.
* **Accuracy vs. reasoning length** — do long chains signal "overthinking" / getting lost?
* **Paired robustness**: for each GSM8K problem, did the answer stay correct under each perturbation?

In [ ]:
# Colour roles (colourblind-safe): blue = correct, orange = incorrect; splits use a fixed categorical order.
C_CORRECT, C_WRONG = "#2a78d6", "#eb6834"
SPLIT_COLORS = {"gsm8k": "#2a78d6", "symbolic": "#eb6834", "symbolic_p1": "#1baf7a", "noop": "#eda100"}
SPLIT_LABELS = {"gsm8k": "GSM8K (original)", "symbolic": "GSM-Symbolic", "symbolic_p1": "GSM-Symbolic P1",
                "noop": "+ irrelevant sentence"}
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 10})

BACKTRACK_PATTERNS = {
    "wait": r"\bwait\b",
    "alternatively": r"\balternatively\b",
    "hmm": r"\bhmm+\b",
    "let me check": r"\blet me (?:double[- ]?)?(?:check|verify|re-?check|re-?calculate|re-?examine)\b",
    "actually": r"\bactually\b",
    "mistake": r"\b(?:mistake|that's not right|that doesn't seem right)\b",
}

def count_backtracking(text: str) -> dict:
    t = text.lower()
    return {k: len(re.findall(p, t)) for k, p in BACKTRACK_PATTERNS.items()}

def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centre - half, centre + half

def savefig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight", dpi=150)

# ---- Load all generations and recompute derived columns.
rows = [r for split in SPLITS_TO_RUN for r in load_jsonl(OUT_DIR / f"generations_{split}.jsonl")]
df = pd.DataFrame(rows)
assert len(df), "No generations found - run Section 6 first."
df[["pred", "extraction"]] = df["output"].apply(lambda o: pd.Series(extract_answer(o)))
df["correct"] = [is_correct(p, g) for p, g in zip(df["pred"], df["gold"])]
bt = pd.DataFrame([count_backtracking(r) for r in df["reasoning"]])
df = pd.concat([df.drop(columns=[c for c in bt.columns if c in df.columns]), bt], axis=1)
df["n_backtrack"] = bt.sum(axis=1)
df["outcome"] = np.where(df["correct"], "correct", "incorrect")
splits_present = [s for s in SPLITS_TO_RUN if s in set(df["split"])]

summary = df.groupby("split").agg(n=("correct", "size"), accuracy=("correct", "mean"),
                                  hit_limit_rate=("hit_limit", "mean"),
                                  median_reasoning_tokens=("n_reasoning_tokens", "median"),
                                  mean_backtrack=("n_backtrack", "mean"),
                                  no_box_rate=("extraction", lambda e: (e != "boxed").mean())).reindex(splits_present)
summary.to_csv(OUT_DIR / "summary_by_split.csv")
display(summary.round(3))

### 7.1 Accuracy: original vs. perturbed

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
for i, s in enumerate(splits_present):
    sub = df[df["split"] == s]
    k, n = sub["correct"].sum(), len(sub)
    lo, hi = wilson_ci(k, n)
    ax.bar(i, k / n, width=0.6, color=SPLIT_COLORS[s])
    ax.errorbar(i, k / n, yerr=[[k / n - lo], [hi - k / n]], color="#333333", capsize=4, lw=1)
    ax.text(i, hi + 0.02, f"{k / n:.0%}\n(n={n})", ha="center", va="bottom", fontsize=9)
ax.set_xticks(range(len(splits_present)), [SPLIT_LABELS[s] for s in splits_present])
ax.set_ylim(0, 1.15)
ax.set_ylabel("Accuracy")
ax.set_title(f"{MODEL_KEY}: accuracy per problem set (95% Wilson CI)")
ax.grid(axis="x", visible=False)
savefig(fig, "accuracy_by_split"); plt.show()

### 7.2 Reasoning length: correct vs. incorrect
Long chains that end wrong are a typical failure mode of small reasoning models. The dashed line is the token budget: chains near it were probably cut off.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
bins = np.linspace(0, max(MAX_NEW_TOKENS, df["n_reasoning_tokens"].max()) + 1, 30)
for outcome, color in [("correct", C_CORRECT), ("incorrect", C_WRONG)]:
    vals = df.loc[df["outcome"] == outcome, "n_reasoning_tokens"]
    ax.hist(vals, bins=bins, alpha=0.6, color=color, edgecolor="white", linewidth=1,
            label=f"{outcome} (n={len(vals)}, median {vals.median():.0f})")
ax.axvline(MAX_NEW_TOKENS, color="#555555", ls="--", lw=1)
ax.text(MAX_NEW_TOKENS, ax.get_ylim()[1] * 0.95, " token limit", fontsize=8, va="top", color="#555555")
ax.set_xlabel("Reasoning length (tokens before </think>)")
ax.set_ylabel("Number of problems")
ax.set_title("Reasoning length by correctness (all problem sets)")
ax.legend(frameon=False)
savefig(fig, "reasoning_length_hist"); plt.show()

### 7.3 Backtracking markers vs. correctness
Left: total markers per problem. Right: average count of each marker type.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [1, 1.6]})

ax = axes[0]
data = [df.loc[df["outcome"] == o, "n_backtrack"] for o in ["correct", "incorrect"]]
bp = ax.boxplot(data, widths=0.5, patch_artist=True, showfliers=False)
for patch, c in zip(bp["boxes"], [C_CORRECT, C_WRONG]):
    patch.set(facecolor=c, alpha=0.35, edgecolor=c)
jitter = np.random.default_rng(SEED)
for i, (vals, c) in enumerate(zip(data, [C_CORRECT, C_WRONG]), start=1):
    ax.scatter(i + jitter.uniform(-0.12, 0.12, len(vals)), vals, s=12, color=c, alpha=0.7)
ax.set_xticks([1, 2], [f"correct (n={len(data[0])})", f"incorrect (n={len(data[1])})"])
ax.set_ylabel("Backtracking markers per problem")
ax.set_title("Total backtracking markers")
ax.grid(axis="x", visible=False)

ax = axes[1]
means = df.groupby("outcome")[list(BACKTRACK_PATTERNS)].mean().reindex(["correct", "incorrect"])
x = np.arange(len(BACKTRACK_PATTERNS))
ax.bar(x - 0.2, means.loc["correct"], width=0.38, color=C_CORRECT, label="correct")
ax.bar(x + 0.2, means.loc["incorrect"], width=0.38, color=C_WRONG, label="incorrect")
ax.set_xticks(x, list(BACKTRACK_PATTERNS), rotation=20)
ax.set_ylabel("Mean count per problem")
ax.set_title("Marker types")
ax.legend(frameon=False)
ax.grid(axis="x", visible=False)
fig.tight_layout()
savefig(fig, "backtracking_vs_correctness"); plt.show()

# Normalised view: markers per 1k reasoning tokens (long chains naturally contain more markers).
df["backtrack_per_1k"] = 1000 * df["n_backtrack"] / df["n_reasoning_tokens"].clip(lower=1)
display(df.groupby("outcome")[["n_backtrack", "backtrack_per_1k"]].describe().round(2))

### 7.4 Accuracy vs. reasoning length ("overthinking")
Problems are binned into quantiles of reasoning length. A falling curve means long chains are a warning sign (the model is lost or loops), not a sign of extra care. Note that length is confounded with problem difficulty — Notebook 02 should control for this (e.g. compare the same problem across samples).

In [ ]:
n_bins = min(5, max(1, len(df) // 10))
df["len_bin"] = pd.qcut(df["n_reasoning_tokens"], q=n_bins, duplicates="drop")
g = df.groupby("len_bin", observed=True)["correct"].agg(["sum", "count"])
centres = df.groupby("len_bin", observed=True)["n_reasoning_tokens"].median()
acc = g["sum"] / g["count"]
cis = np.array([wilson_ci(k, n) for k, n in zip(g["sum"], g["count"])])

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.fill_between(centres, cis[:, 0], cis[:, 1], color=C_CORRECT, alpha=0.15, linewidth=0)
ax.plot(centres, acc, "-o", color=C_CORRECT, lw=2, ms=6)
for xc, a, n in zip(centres, acc, g["count"]):
    ax.annotate(f"n={n}", (xc, a), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8)
ax.set_xlabel("Reasoning length (median tokens of quantile bin)")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1.05)
ax.set_title("Accuracy vs. reasoning length (quantile bins, 95% CI)")
savefig(fig, "accuracy_vs_length"); plt.show()

### 7.5 Paired robustness: does a perturbation break a problem the model solved?
For every GSM8K problem with a perturbed twin we count four outcomes: *stays correct*, *breaks* (correct → wrong), *fixed* (wrong → correct) and *stays wrong*. With sampling, some flips are just randomness — Notebook 02 should use several samples per problem to separate noise from real sensitivity.

In [ ]:
orig = df[df["split"] == "gsm8k"].set_index("gsm8k_id")["correct"]
pair_rows = []
for s in [s for s in splits_present if s != "gsm8k"]:
    pert = df[df["split"] == s].set_index("gsm8k_id")["correct"]
    common = orig.index.intersection(pert.index)
    o, p = orig.loc[common], pert.loc[common]
    pair_rows.append(dict(split=s, n=len(common), stays_correct=int((o & p).sum()), breaks=int((o & ~p).sum()),
                          fixed=int((~o & p).sum()), stays_wrong=int((~o & ~p).sum())))
pairs = pd.DataFrame(pair_rows)

if len(pairs):
    display(pairs)
    pairs.to_csv(OUT_DIR / "paired_robustness.csv", index=False)
    fig, ax = plt.subplots(figsize=(7, 0.8 + 0.6 * len(pairs)))
    cats = [("stays_correct", "#2a78d6", "stays correct"), ("breaks", "#eb6834", "breaks (✓→✗)"),
            ("fixed", "#1baf7a", "fixed (✗→✓)"), ("stays_wrong", "#bdbdbd", "stays wrong")]
    left = np.zeros(len(pairs))
    for col, color, label in cats:
        frac = pairs[col] / pairs["n"]
        ax.barh(range(len(pairs)), frac, left=left, color=color, edgecolor="white", linewidth=2, label=label)
        left += frac
    ax.set_yticks(range(len(pairs)), [SPLIT_LABELS[s] for s in pairs["split"]])
    ax.set_xlim(0, 1)
    ax.set_xlabel("Share of paired problems")
    ax.set_title("GSM8K original vs. perturbed twin")
    ax.legend(frameon=False, ncol=4, bbox_to_anchor=(0.5, -0.35 - 0.1 * len(pairs)), loc="upper center")
    ax.grid(False)
    savefig(fig, "paired_robustness"); plt.show()
else:
    print("No perturbed splits available.")

In [ ]:
# A look at one failure: the shortest incorrect chain (easiest to read).
wrong = df[~df["correct"]].sort_values("n_reasoning_tokens")
if len(wrong):
    r = wrong.iloc[0]
    print(f"[{r['uid']}] gold={r['gold']} pred={r['pred']} tokens={r['n_tokens']} hit_limit={r['hit_limit']}\n")
    print("QUESTION:", r["question"], "\n")
    print("REASONING (first 2500 chars):\n", r["reasoning"][:2500])
    print("\nFINAL:", r["final_text"][:500])

## 8. What did the model learn? — Early answering

**Idea:** take a finished reasoning chain, cut it after 0%, 10%, …, 100% of its length, then force the model to answer immediately:

```
<prompt><think>\n  [first x% of the reasoning]  \n</think>\n\nThe final answer is \boxed{
```

With **one forward pass** we read the probability the model assigns to the **gold answer** (all its digits plus the closing `}`, via teacher forcing). We also do a short greedy decode to see *which* answer it would give at that point.
If the probability jumps to ~1 early, the model "knew" the answer long before it stopped reasoning (the rest is verification or post-hoc rationalisation). If it only rises at the very end, the reasoning is actually doing the work.

Cuts snap to the nearest whitespace so we never split a number in the middle. We use ~10 GSM8K problems, mixing correct and incorrect ones. Results are appended to Drive and skipped on re-run.

In [ ]:
THINK_CLOSE = "\n</think>\n\n" if IS_REASONING_MODEL else "\n\n"
ANSWER_PREFIX = "The final answer is \\boxed{"
decoder = model.get_decoder()            # transformer body (returns normalised hidden states)
lm_head = model.get_output_embeddings()  # hidden state -> vocabulary logits


def cut_reasoning(reasoning: str, frac: float) -> str:
    # First `frac` of the reasoning (by characters), snapped back to the previous whitespace.
    if frac >= 1:
        return reasoning
    cut = int(len(reasoning) * frac)
    while cut > 0 and not reasoning[cut].isspace():
        cut -= 1
    return reasoning[:cut]


def early_answer_context(question: str, reasoning_prefix: str) -> str:
    return build_prompt(question) + reasoning_prefix.rstrip() + THINK_CLOSE + ANSWER_PREFIX


@torch.no_grad()
def answer_logprob(ctx_ids, answer: str):
    # log P(answer + '}' | context) with a single teacher-forced forward pass.
    tgt_ids = tokenizer(answer + "}", add_special_tokens=False).input_ids
    ids = torch.tensor([ctx_ids + tgt_ids[:-1]], device=DEVICE)
    hidden = decoder(input_ids=ids, use_cache=False).last_hidden_state[0]
    positions = torch.arange(len(ctx_ids) - 1, len(ctx_ids) - 1 + len(tgt_ids), device=DEVICE)
    logprobs = torch.log_softmax(lm_head(hidden[positions]).float(), dim=-1)
    token_lp = logprobs[torch.arange(len(tgt_ids)), torch.tensor(tgt_ids, device=DEVICE)]
    return token_lp.sum().item(), token_lp[0].exp().item()      # (joint log-prob, P(first token))


@torch.no_grad()
def greedy_answer(ctx_ids, max_tokens=12):
    ids = torch.tensor([ctx_ids], device=DEVICE)
    out = model.generate(input_ids=ids, attention_mask=torch.ones_like(ids), max_new_tokens=max_tokens,
                         do_sample=False, temperature=None, top_p=None, top_k=None,
                         pad_token_id=tokenizer.pad_token_id)
    text = tokenizer.decode(out[0, len(ctx_ids):], skip_special_tokens=True)
    return text.split("}")[0].strip()

In [ ]:
# ---- Pick problems: GSM8K split, completed chains preferred, half correct / half incorrect where possible.
base_split = "gsm8k" if "gsm8k" in splits_present else splits_present[0]
cand = df[df["split"] == base_split].sample(frac=1.0, random_state=SEED)
cand = cand.sort_values("think_closed", ascending=False, kind="stable")   # finished chains first
n_wrong = min(N_EARLY // 2, int((~cand["correct"]).sum()))
early_set = pd.concat([cand[~cand["correct"]].head(n_wrong), cand[cand["correct"]].head(N_EARLY - n_wrong)])
early_set = pd.concat([early_set, cand.drop(early_set.index)]).head(N_EARLY)    # top up if one group is small
print(f"Early answering on {len(early_set)} problems "
      f"({int(early_set['correct'].sum())} correct, {int((~early_set['correct']).sum())} incorrect)")

EARLY_PATH = OUT_DIR / "early_answering.jsonl"
done_keys = {(r["uid"], r["frac"]) for r in load_jsonl(EARLY_PATH)}

for _, r in early_set.iterrows():
    gold_str = format_number(to_number(r["gold"]))
    pred_str = format_number(r["pred"]) if pd.notna(r["pred"]) else None
    todo = [f for f in EARLY_FRACTIONS if (r["uid"], f) not in done_keys]
    if not todo:
        continue
    t0, recs = time.time(), []
    for frac in todo:
        prefix = cut_reasoning(r["reasoning"], frac)
        ctx_ids = tokenizer(early_answer_context(r["question"], prefix), add_special_tokens=False).input_ids
        lp_gold, p_first_gold = answer_logprob(ctx_ids, gold_str)
        lp_pred = answer_logprob(ctx_ids, pred_str)[0] if pred_str not in (None, gold_str) else lp_gold
        greedy = greedy_answer(ctx_ids)
        recs.append(dict(uid=r["uid"], frac=frac, gold=gold_str, final_pred=pred_str, final_correct=bool(r["correct"]),
                         n_prefix_tokens=len(tokenizer(prefix, add_special_tokens=False).input_ids),
                         p_gold=math.exp(lp_gold), p_gold_first_token=p_first_gold,
                         p_final_pred=math.exp(lp_pred) if pred_str else None,
                         greedy_answer=greedy, greedy_correct=is_correct(to_number(greedy), gold_str)))
    append_jsonl(EARLY_PATH, recs)          # saved per problem
    print(f"  {r['uid']}: {len(recs)} cut points in {time.time() - t0:.0f}s")

early = pd.DataFrame(load_jsonl(EARLY_PATH))
early = early[early["uid"].isin(early_set["uid"])]
early.to_csv(OUT_DIR / "early_answering.csv", index=False)
display(early.head())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1.3, 1]})

# Left: P(gold answer) vs. % of reasoning, one line per problem (blue = final answer correct, orange = wrong).
ax = axes[0]
for uid, g in early.groupby("uid"):
    g = g.sort_values("frac")
    ok = g["final_correct"].iloc[0]
    ax.plot(100 * g["frac"], g["p_gold"], "-o", ms=4, lw=1.6, alpha=0.8, color=C_CORRECT if ok else C_WRONG)
ax.plot([], [], color=C_CORRECT, lw=2, label="final answer correct")
ax.plot([], [], color=C_WRONG, lw=2, label="final answer wrong")
ax.set_xlabel("% of the model's reasoning shown before forcing an answer")
ax.set_ylabel("P(gold answer)")
ax.set_ylim(-0.03, 1.03)
ax.set_title("Early answering: when does the model 'know' the answer?")
ax.legend(frameon=False, loc="upper left")

# Right: does a greedy forced answer at that point already equal the gold answer?
ax = axes[1]
order = early.groupby("uid")["final_correct"].first().sort_values(ascending=False).index
mat = early.pivot(index="uid", columns="frac", values="greedy_correct").loc[order].astype(float)
ax.imshow(mat.values, aspect="auto", cmap="Blues", vmin=0, vmax=1.4, interpolation="nearest")
ax.set_xticks(range(mat.shape[1]), [f"{int(100 * f)}" for f in mat.columns])
ax.set_yticks(range(mat.shape[0]), [f"{u}{'' if early_set.set_index('uid').loc[u, 'correct'] else ' ✗'}"
                                    for u in mat.index], fontsize=8)
ax.set_xlabel("% of reasoning")
ax.set_title("Greedy forced answer = gold (dark)\n✗ = final answer wrong")
ax.grid(False)
fig.tight_layout()
savefig(fig, "early_answering"); plt.show()

# Summary: earliest cut at which the greedy forced answer is already correct (and stays correct).
def first_stable_correct(g):
    g = g.sort_values("frac")
    vals = g["greedy_correct"].values
    for i in range(len(vals)):
        if vals[i:].all():
            return g["frac"].values[i]
    return np.nan
print("Earliest % of reasoning after which the forced answer stays correct:")
display((100 * early.groupby("uid").apply(first_stable_correct)).rename("pct").to_frame())

## 9. Optional: logit lens — in which layer does the answer appear?

The **logit lens** applies the model's final RMSNorm + `lm_head` to the hidden state after each intermediate layer, as if that layer were the last one. We do this at the answer position (right after `\boxed{`) for the same early-answering contexts and plot a **layer × reasoning-progress heatmap** of the probability of the gold answer's **first token**.

**Note on tokenisation:** Qwen tokenises numbers **digit by digit**, so the first token of `18` is just `1`. A high probability for the first digit is necessary but not sufficient for knowing the full answer (`1` could start `12` or `150`). Section 8 scores the whole number; this view adds *where in the network* the first digit becomes decodable.

Hidden states are captured with forward hooks on each decoder layer (this works across `transformers` versions); only one position per layer is stored, so memory use is small.

In [ ]:
@torch.no_grad()
def logit_lens_first_token(ctx_ids, first_token_id):
    captured = []
    def hook(module, inputs, output):
        h = output[0] if isinstance(output, (tuple, list)) else output
        captured.append(h[0, -1].detach())
    handles = [layer.register_forward_hook(hook) for layer in decoder.layers]
    try:
        decoder(input_ids=torch.tensor([ctx_ids], device=DEVICE), use_cache=False)
    finally:
        for hd in handles:
            hd.remove()
    H = torch.stack(captured)                                   # (n_layers, hidden)
    probs = torch.softmax(lm_head(decoder.norm(H)).float(), dim=-1)
    return probs[:, first_token_id].cpu().numpy()               # (n_layers,)


if RUN_LOGIT_LENS:
    LENS_PATH = OUT_DIR / "logit_lens.jsonl"
    lens_done = {r["uid"] for r in load_jsonl(LENS_PATH)}
    lens_set = early_set.head(N_LOGIT_LENS)
    for _, r in lens_set.iterrows():
        if r["uid"] in lens_done:
            continue
        gold_str = format_number(to_number(r["gold"]))
        first_id = tokenizer(gold_str, add_special_tokens=False).input_ids[0]
        grid = []
        for frac in EARLY_FRACTIONS:
            ctx_ids = tokenizer(early_answer_context(r["question"], cut_reasoning(r["reasoning"], frac)),
                                add_special_tokens=False).input_ids
            grid.append(logit_lens_first_token(ctx_ids, first_id).tolist())
        append_jsonl(LENS_PATH, [dict(uid=r["uid"], gold=gold_str, first_token=tokenizer.decode([first_id]),
                                      final_correct=bool(r["correct"]), fracs=EARLY_FRACTIONS,
                                      probs=grid)])           # probs[frac_index][layer]
        print(f"  logit lens done: {r['uid']}")

    lens = [x for x in load_jsonl(LENS_PATH) if x["uid"] in set(lens_set["uid"])]
    if lens:
        fig, axes = plt.subplots(1, len(lens), figsize=(3.6 * len(lens), 4), sharey=True, squeeze=False)
        for ax, x in zip(axes[0], lens):
            P = np.array(x["probs"]).T                              # (layers, fractions)
            im = ax.imshow(P, origin="lower", aspect="auto", cmap="Blues", vmin=0, vmax=1)
            ax.set_xticks(range(0, len(x["fracs"]), 2), [f"{int(100 * f)}" for f in x["fracs"][::2]])
            ax.set_xlabel("% of reasoning")
            ax.set_title(f"{x['uid']}\ngold {x['gold']} (first token '{x['first_token']}')"
                         f"{'' if x['final_correct'] else ' ✗'}", fontsize=9)
            ax.grid(False)
        axes[0][0].set_ylabel("Layer")
        fig.colorbar(im, ax=axes[0].tolist(), label="P(first gold token)", shrink=0.85)
        fig.suptitle("Logit lens at the answer position", y=1.02)
        savefig(fig, "logit_lens"); plt.show()

## 10. Where results are, and what comes next

**Saved on Drive** (`OUT_DIR`): `config.json`, `generations_<split>.jsonl`, `summary_by_split.csv`, `paired_robustness.csv`, `early_answering.jsonl/.csv`, `logit_lens.jsonl`, and all figures in `figures/`.

### Suggested next notebooks
1. **Notebook 02 — Baseline comparison.** Set `MODEL_KEY = "qwen2.5-math-1.5b"` and re-run the same splits. Compare accuracy, chain length and backtracking with the R1 distill. (Qwen2.5-Math-1.5B is a base model without a `<think>` block; check that its outputs end cleanly, consider a lower `MAX_NEW_TOKENS` and a stop rule, and note that "reasoning" = the whole output there.)
2. **Scale up and repeat.** Increase `N_PROBLEMS` (e.g. 200–500 GSM8K problems) and draw several samples per problem (different seeds) to separate randomness from real failures; report per-problem pass rates.
3. **More perturbations.** All 50 GSM-Symbolic instances per template (accuracy *distribution* per template, as in Apple's paper), P2, name-only vs. number-only changes, and a stronger NoOp set with *seemingly relevant* distractors.
4. **Deeper "what did it learn".** Early answering on all problems, per-sentence cut points, aligning probability jumps with backtracking markers ("does the answer change after a *wait*?"), and linear probes on hidden states that predict the final answer / correctness.
5. **Better visualisations.** An interactive explorer (e.g. Plotly or a small web app) that shows a reasoning chain with backtracking markers highlighted, coloured by the early-answering probability at each point, plus side-by-side original/perturbed chains.